# De Novo Protein Binder Design with NVIDIA BioNeMo NIMs

Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

**Optional stream.** This notebook is a self-contained workflow that is independent of the
MolMIM / CDK inhibitor track. It walks through an end-to-end **de novo protein binder design**
campaign by composing three NVIDIA BioNeMo NIMs — **RFdiffusion → ProteinMPNN → Boltz-2** —
to design a brand-new protein that binds a chosen target at a chosen epitope, then validates
and ranks the designs.

It is tuned to finish in **well under an hour** on a GB200 (Grace-Blackwell) node in demo mode.

## 1. What is de novo protein binder design?

A **binder** is a protein that sticks to a target protein. *De novo* design means we invent a
**completely new** protein (not found in nature) that binds the target at a specific site
(**epitope / hotspots**) — rather than screening natural antibodies or libraries.

Why it matters:

- **Therapeutics** — binders can block a disease-relevant interaction (e.g. a virus binding its
  human receptor), neutralize a toxin, or act as targeted delivery agents. Small de novo
  "minibinders" are stable, cheap to manufacture, and easy to engineer.
- **Diagnostics & research tools** — high-affinity binders are reagents for assays and imaging.
- **Speed** — generative models propose candidates in minutes that would take months of wet-lab
  screening, focusing experiments on a small, high-confidence shortlist.

The modern computational recipe is: **generate a backbone shape → design a sequence for it →
predict the complex to check it actually binds → rank.** Each step is an NVIDIA NIM.

## 2. Target: the SARS-CoV-2 spike Receptor-Binding Domain (RBD)

We design binders against the **Receptor-Binding Domain (RBD)** of the SARS-CoV-2 spike protein.

**Biology.** The RBD is the part of the spike that grabs the human receptor **ACE2** to enter
cells. The RBD↔ACE2 interface is therefore the *Achilles' heel* of the virus: a binder that
covers that interface physically **blocks viral entry** (it competes with ACE2).

**Therapeutic relevance.**

- Antibodies and de novo **minibinders** that block RBD↔ACE2 are *neutralizing* — they stop
  infection. In a landmark study (Cao et al., *Science* 2020), computationally designed
  mini-protein binders to this exact interface neutralized SARS-CoV-2, proving de novo binders
  can be real therapeutic leads.
- The same interface is shared across sarbecoviruses, motivating **broad / pandemic-preparedness**
  binders, plus diagnostics and research reagents.

**Structure & epitope.** We use **PDB `6M0J`** (RBD, chain `E`, crystallized with ACE2) and target
the canonical ACE2-contact **hotspot residues** `453, 455, 456, 486, 489, 493, 501` (author
numbering). Conditioning RFdiffusion on these hotspots steers binders to the ACE2 site.

> Educational example. Verify epitopes/structures against the literature before any real campaign.

## 3. The NVIDIA BioNeMo NIMs in this workflow

| NIM | Role | Input → Output |
|---|---|---|
| **RFdiffusion** | *Backbone generation* — a diffusion model that "grows" a binder **3D backbone** against the target, steered by the hotspots. | target PDB + contigs + hotspots → binder **backbone** PDB |
| **ProteinMPNN** | *Inverse folding* — designs **amino-acid sequences** predicted to fold into a given backbone (interface-aware). | backbone PDB → **sequences** (+ scores) |
| **Boltz-2** | *Co-folding / validation* — predicts the 3D structure of the **binder + target complex** and reports confidence; we use it as an in-silico binding test. | binder seq + target seq → **complex** (mmCIF) + confidence / pLDDT |

Notes:

- **Boltz-2** is the same NIM the CDK track uses for affinity; here we use its **structure-prediction**
  endpoint to *co-fold* a 2-chain complex. **OpenFold3** is a drop-in alternative co-folder that
  additionally reports an explicit **ipTM** interface score.
- All three are reached over HTTP and run either **hosted** (build.nvidia.com) or **locally**
  (self-hosted NIM containers). On **GB200/GB300**, all three have **arm64** images, so the whole
  pipeline can run locally on the node.

## 4. Workflow architecture

```mermaid
flowchart LR
    T["Target RBD (6M0J:E)
+ ACE2 hotspots"] --> R["RFdiffusion
N backbones"]
    R --> P["ProteinMPNN
k sequences / backbone"]
    P --> B["Boltz-2 co-fold
complex + confidence + pLDDT"]
    B --> S["Self-consistency
CA-RMSD vs backbone"]
    S --> F["Filter + rank
ipTM / pLDDT / RMSD"]
    F --> O["Ranked binders
manifest.json + candidates.csv"]
```

This is a **cost funnel**: generate many cheap backbones and sequences, then spend the expensive
co-folding step only on a **shortlist**. We also co-fold **scrambled-sequence negative controls**
so we can report a **success rate** (designs beating controls), not just top scores.

**Filters (defaults):** interface confidence (ipTM / Boltz-2 confidence) ≥ 0.8, binder pLDDT ≥ 80,
self-consistency CA-RMSD ≤ 2.0 Å.

## 5. Before you run

**NIM endpoints.** Set these environment variables before launching Jupyter (defaults are the
NVIDIA-hosted endpoints, which need `NVIDIA_API_KEY`):

| Variable | Purpose | Default |
|---|---|---|
| `RFDIFFUSION_URL` | RFdiffusion base URL | hosted |
| `PROTEINMPNN_URL` | ProteinMPNN base URL | hosted |
| `BOLTZ2_URL` | Boltz-2 base URL | hosted (or the bootcamp's local Boltz-2) |
| `NVIDIA_API_KEY` | bearer token for hosted NIMs | — |

- **Hosted:** `export NVIDIA_API_KEY=nvapi-...` and leave the URLs at their defaults.
- **Local on GB200:** launch the three NIMs (they have arm64 images) and point the URLs at them,
  e.g. `http://localhost:8081` (RFdiffusion), `http://localhost:8082` (ProteinMPNN),
  `http://localhost:8000` (Boltz-2). See `protein-binder-design/README.md`.

**Runtime budget.** `OPENHACKATHON_DEMO_MODE=1` (default) keeps the campaign small (a few
backbones / sequences and a short co-fold shortlist) so it finishes in **< 1 hour**. Set it to `0`
for a fuller campaign.

**Responsible use.** De novo binder design is dual-use. Use it only for legitimate research and
therapeutic intent.

In [ ]:
import os, sys, json, time, pathlib, datetime, subprocess

# Optional 3D viewers: Mol* (ipymolstar) for complexes, py3Dmol for simple views.
for pkg in ("ipymolstar", "py3Dmol"):
    try:
        __import__(pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

# Make the local binder_design package importable (notebook runs from its folder).
HERE = pathlib.Path.cwd()
for cand in (HERE, HERE / "protein-binder-design"):
    if (cand / "binder_design").exists():
        sys.path.insert(0, str(cand)); break

from binder_design import nim_clients, pdb_utils, metrics, viz
from binder_design.manifest import Manifest
from binder_design.controls import make_scrambled_controls

# ---- configuration ---------------------------------------------------------
DEMO_MODE = os.environ.get("OPENHACKATHON_DEMO_MODE", "1") != "0"
N_BACKBONES       = int(os.environ.get("PBD_N_BACKBONES", "12" if DEMO_MODE else "24"))
SEQS_PER_BACKBONE = int(os.environ.get("PBD_SEQS_PER_BACKBONE", "2" if DEMO_MODE else "4"))
N_COFOLD          = int(os.environ.get("PBD_N_COFOLD", "8" if DEMO_MODE else "16"))
N_CONTROLS        = int(os.environ.get("PBD_N_CONTROLS", "3" if DEMO_MODE else "4"))
DIFFUSION_STEPS   = int(os.environ.get("PBD_DIFFUSION_STEPS", "30" if DEMO_MODE else "50"))
COFOLD_STEPS      = int(os.environ.get("PBD_COFOLD_STEPS", "50"))
BINDER_MIN, BINDER_MAX = 55, 90

RFDIFFUSION_URL = os.environ.get("RFDIFFUSION_URL", nim_clients.HOSTED_ROOT)
PROTEINMPNN_URL = os.environ.get("PROTEINMPNN_URL", nim_clients.HOSTED_ROOT)
BOLTZ2_URL      = os.environ.get("BOLTZ2_URL", nim_clients.HOSTED_ROOT)

print(f"DEMO_MODE={DEMO_MODE}  backbones={N_BACKBONES}  seqs/bb={SEQS_PER_BACKBONE} "
      f"co-fold={N_COFOLD}  controls={N_CONTROLS}  diffusion_steps={DIFFUSION_STEPS}  "
      f"cofold_steps={COFOLD_STEPS}")

In [ ]:
rfd  = nim_clients.RFdiffusionClient(RFDIFFUSION_URL)
mpnn = nim_clients.ProteinMPNNClient(PROTEINMPNN_URL)
b2   = nim_clients.Boltz2StructureClient(BOLTZ2_URL)

for name, c in [("RFdiffusion", rfd), ("ProteinMPNN", mpnn), ("Boltz-2", b2)]:
    mode = "hosted" if "api.nvidia.com" in c.base else "local"
    try:
        ok = nim_clients.health_ready(c.base)
    except Exception as e:
        ok = f"error: {e}"
    print(f"{name:12s} [{mode:6s}] {c.url}\n             ready={ok}")

## 6. Step 1 — Target preparation

Download `6M0J`, isolate the RBD (chain `E`), read its sequence, and map the ACE2 hotspot
residues. RFdiffusion uses chain+author **hotspot strings** (e.g. `"E453"`); other tools use
**1-based sequence indices**, so we compute both.

In [ ]:
PDB_ID = "6M0J"
TARGET_CHAIN = "E"
HOTSPOTS_AUTHOR = [453, 455, 456, 486, 489, 493, 501]   # ACE2-contact residues on the RBD

pdb_text = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=60).text
target_pdb = pdb_utils.extract_chain(pdb_text, TARGET_CHAIN)
target_seq = pdb_utils.sequence(target_pdb, TARGET_CHAIN)
hotspot_seq_idx = pdb_utils.remap_to_seq_index(target_pdb, TARGET_CHAIN, HOTSPOTS_AUTHOR)
hotspot_res = [f"{TARGET_CHAIN}{a}" for a in HOTSPOTS_AUTHOR]

resn = pdb_utils.ca_residues(target_pdb, TARGET_CHAIN)
first_res, last_res = resn[0][1], resn[-1][1]
contigs = f"{TARGET_CHAIN}{first_res}-{last_res}/0 {BINDER_MIN}-{BINDER_MAX}"

print(f"Target: SARS-CoV-2 RBD ({PDB_ID} chain {TARGET_CHAIN}), {len(target_seq)} residues "
      f"(author {first_res}-{last_res})")
print(f"Hotspots (author): {hotspot_res}")
print(f"Hotspots (1-based seq idx): {hotspot_seq_idx}")
print(f"RFdiffusion contigs: {contigs!r}  (keep RBD, generate {BINDER_MIN}-{BINDER_MAX} aa binder)")
print(f"Target sequence:\n{target_seq}")

### Visualize the target and its hotspots

The RBD is shown as a light-blue cartoon; the ACE2-contact **hotspots are red** — this is the
surface patch our binders should cover.

In [ ]:
view = viz.show_target_with_hotspots(target_pdb, TARGET_CHAIN, HOTSPOTS_AUTHOR)
view.show()

## 7. Step 2 — Generate binder backbones (RFdiffusion)

RFdiffusion diffuses a binder **backbone** against the RBD, steered toward the hotspots. We create
a run directory + manifest (for reproducibility), then generate `N_BACKBONES` backbones. For each
backbone we auto-detect the **binder chain** (the shorter chain; the longer one is the RBD).

In [ ]:
run_dir = pathlib.Path("runs") / f"{PDB_ID}_{datetime.datetime.now():%Y%m%d_%H%M%S}"
m = Manifest.create(
    run_dir=run_dir,
    target={"name": "SARS-CoV-2 RBD", "pdb_id": PDB_ID, "chain": TARGET_CHAIN,
            "hotspots_author": HOTSPOTS_AUTHOR},
    mode="hosted" if "api.nvidia.com" in rfd.base else "local",
    params={"n_backbones": N_BACKBONES, "seqs_per_backbone": SEQS_PER_BACKBONE,
            "binder_len": f"{BINDER_MIN}-{BINDER_MAX}", "diffusion_steps": DIFFUSION_STEPS},
)
bb_dir = run_dir / "backbones"; bb_dir.mkdir(parents=True, exist_ok=True)

backbones = {}
t0 = time.time()
for i in range(N_BACKBONES):
    bid = f"bb{i:03d}"
    try:
        bb_pdb = rfd.generate(target_pdb, contigs, hotspot_res, diffusion_steps=DIFFUSION_STEPS)
    except Exception as e:
        print(f"  {bid}: RFdiffusion failed: {e}"); continue
    chains = pdb_utils.chain_ids(bb_pdb)
    lens = {c: len(pdb_utils.sequence(bb_pdb, c)) for c in chains}
    binder_chain = min(lens, key=lens.get)            # binder is the shorter chain
    (bb_dir / f"{bid}.pdb").write_text(bb_pdb)
    backbones[bid] = {"pdb": bb_pdb, "binder_chain": binder_chain, "binder_len": lens[binder_chain]}
    m.upsert_candidate(bid, backbone_id=bid)
    m.add_artifact(bid, "backbone_pdb", str(bb_dir / f"{bid}.pdb"))
    print(f"  {bid}: chains={chains} lens={lens} binder={binder_chain}({lens[binder_chain]} aa)")

m.log_stage("backbones", n=len(backbones), seconds=round(time.time() - t0, 1))
print(f"\nGenerated {len(backbones)} backbones in {time.time()-t0:.0f}s -> {bb_dir}")

### Visualize a generated backbone

A representative RFdiffusion backbone: the small chain is the **de novo binder**, docked against
the RBD.

In [ ]:
if backbones:
    bid0 = next(iter(backbones))
    v = viz.show_backbone(backbones[bid0]["pdb"])
    v.show()

## 8. Step 3 — Design sequences (ProteinMPNN)

For each backbone we ask ProteinMPNN for `SEQS_PER_BACKBONE` sequences, **redesigning only the
binder chain** (the RBD is held fixed so the interface is designed in context). ProteinMPNN
returns multi-chain rows joined by `/`; we extract the binder segment and drop the native row.
Lower ProteinMPNN score (NLL) ≈ better fit to the backbone.

In [ ]:
designs = []
t0 = time.time()
for bid, bb in backbones.items():
    chain_order = pdb_utils.chain_ids(bb["pdb"])
    try:
        rows = mpnn.predict(bb["pdb"], input_pdb_chains=[bb["binder_chain"]],
                            num_seq_per_target=SEQS_PER_BACKBONE, sampling_temp=[0.1])
    except Exception as e:
        print(f"  {bid}: ProteinMPNN failed: {e}"); continue
    for j, (header, raw, score) in enumerate(rows):
        binder_seq = nim_clients.binder_subsequence(raw, chain_order, bb["binder_chain"])
        if not binder_seq:
            continue
        did = f"{bid}_s{j:02d}"
        designs.append({"id": did, "backbone_id": bid, "binder_chain": bb["binder_chain"],
                        "seq": binder_seq, "mpnn_score": score})
        m.upsert_candidate(did, backbone_id=bid, sequence=binder_seq)
        m.set_scores(did, proteinmpnn_nll=score)

m.log_stage("sequences", n=len(designs), seconds=round(time.time() - t0, 1))
print(f"Designed {len(designs)} sequences from {len(backbones)} backbones in {time.time()-t0:.0f}s")
for d in designs[:5]:
    print(f"  {d['id']}  nll={d['mpnn_score']}  len={len(d['seq'])}  {d['seq'][:50]}...")

## 9. Step 4 — Co-fold and score (Boltz-2)

Co-folding is the expensive, decisive step, so we only run it on a **shortlist** (best ProteinMPNN
scores) plus **scrambled-sequence negative controls**. For each, Boltz-2 predicts the
**binder + target complex** and we record:

- **interface confidence** (Boltz-2 `confidence` / ipTM proxy),
- **binder pLDDT** (mean over the binder chain; per-residue confidence),
- **self-consistency CA-RMSD** (does the designed sequence fold back onto the RFdiffusion backbone?).

In [ ]:
shortlist = sorted([d for d in designs if d["mpnn_score"] is not None],
                   key=lambda d: d["mpnn_score"])[:N_COFOLD]
if not shortlist:
    shortlist = designs[:N_COFOLD]

ctrl_seqs = make_scrambled_controls([d["seq"] for d in shortlist], n=N_CONTROLS)
controls = [{"id": f"ctrl{i:02d}", "seq": s, "is_control": True} for i, s in enumerate(ctrl_seqs)]
records = shortlist + controls

cx_dir = run_dir / "complexes"; cx_dir.mkdir(parents=True, exist_ok=True)
print(f"Co-folding {len(shortlist)} designs + {len(controls)} controls with Boltz-2...\n")
t0 = time.time()
for d in records:
    tag = "ctrl" if d.get("is_control") else "design"
    try:
        res = b2.predict_complex([("A", d["seq"]), ("B", target_seq)],
                                 recycling_steps=3, sampling_steps=COFOLD_STEPS, diffusion_samples=1)
    except Exception as e:
        print(f"  [{tag}] {d['id']}: Boltz-2 failed: {e}"); continue
    cif = res.get("cif")
    if not cif:
        print(f"  [{tag}] {d['id']}: no structure returned (keys={res.get('raw_keys')})"); continue
    (cx_dir / f"{d['id']}.cif").write_text(cif)
    # binder chain in the predicted complex = chain whose CA count is closest to the binder length
    cif_chains = pdb_utils.mmcif_chain_ids(cif)
    bch = min(cif_chains, key=lambda c: abs(len(pdb_utils.mmcif_ca_coords(cif, c)) - len(d["seq"])))
    tch = next((c for c in cif_chains if c != bch), bch)
    plddt = pdb_utils.mmcif_mean_plddt(cif, bch)
    if plddt <= 1.0:          # normalize 0-1 pLDDT to 0-100
        plddt *= 100.0
    conf = res.get("confidence")
    iptm = res.get("iptm") if res.get("iptm") is not None else conf
    rmsd = None
    if not d.get("is_control"):
        bb = backbones[d["backbone_id"]]
        ref = pdb_utils.ca_coords(pdb_utils.extract_chain(bb["pdb"], bb["binder_chain"]),
                                  bb["binder_chain"])
        pred = pdb_utils.mmcif_ca_coords(cif, bch)
        try:
            rmsd = metrics.ca_rmsd(pred, ref)
        except Exception:
            rmsd = None
    d.update({"iptm": iptm, "binder_plddt": plddt, "boltz2_confidence": conf,
              "self_consistency_rmsd": rmsd, "cif_path": str(cx_dir / f"{d['id']}.cif"),
              "binder_cif_chain": bch, "target_cif_chain": tch})
    if d.get("is_control"):
        m.upsert_candidate(d["id"], is_control=True, control_type="scrambled", sequence=d["seq"])
    m.set_scores(d["id"], iptm=iptm, binder_plddt=plddt, boltz2_confidence=conf,
                 self_consistency_rmsd=rmsd)
    m.add_artifact(d["id"], "complex_cif", d["cif_path"])
    rs = f"{rmsd:.2f}" if rmsd is not None else "  - "
    ic = f"{iptm:.3f}" if iptm is not None else "  -  "
    print(f"  [{tag}] {d['id']}: iptm/conf={ic}  binder_pLDDT={plddt:5.1f}  scRMSD={rs}")

m.log_stage("cofold", n=len(records), seconds=round(time.time() - t0, 1))
print(f"\nCo-folded {len(records)} complexes in {time.time()-t0:.0f}s -> {cx_dir}")

## 10. Step 5 — Visualize a designed binder–target complex (Mol*)

The top design's predicted complex, rendered interactively with **Mol*** (`ipymolstar`):
the **binder is orange**, the **RBD is gray**, and the **ACE2 hotspots are red**. A good design
wraps its binder over the red hotspot patch.

In [ ]:
scored = [d for d in records if not d.get("is_control") and d.get("iptm") is not None]
best = max(scored, key=lambda d: d["iptm"]) if scored else None
if best:
    print(f"Top design: {best['id']}  iptm/conf={best['iptm']:.3f}  binder_pLDDT={best['binder_plddt']:.1f}")
    cif_text = pathlib.Path(best["cif_path"]).read_text()
    mol_view = viz.molstar_complex(
        cif_text, binder_chain=best["binder_cif_chain"], target_chain=best["target_cif_chain"],
        hotspot_seq_idx=hotspot_seq_idx,
    )
else:
    print("No scored designs to visualize.")
    mol_view = None
mol_view

## 11. Step 6 — Filter, rank, and assess

Apply the confidence/pLDDT/RMSD filters, rank the survivors, and compare designs against the
scrambled controls. The **success rate** (designs that beat the filters) is the honest headline
number — top scores alone can be misleading.

In [ ]:
m.apply_filters()
rows = []
for d in records:
    rows.append({
        "id": d["id"],
        "type": "control" if d.get("is_control") else "design",
        "mpnn_nll": d.get("mpnn_score"),
        "iptm": d.get("iptm"),
        "binder_pLDDT": d.get("binder_plddt"),
        "scRMSD": d.get("self_consistency_rmsd"),
        "len": len(d.get("seq", "")),
    })
df = pd.DataFrame(rows).sort_values(["type", "iptm"], ascending=[True, False]).reset_index(drop=True)
pd.set_option("display.width", 140)
print(df.to_string(index=False))

### Assessment plots

(1) the design funnel, (2) interface confidence vs binder pLDDT (designs vs controls, with
filter cutoffs), (3) self-consistency RMSD, and (4) the ranked designs.

In [ ]:
des = [d for d in records if not d.get("is_control") and d.get("iptm") is not None
       and d.get("binder_plddt") is not None]
ctl = [d for d in records if d.get("is_control") and d.get("iptm") is not None
       and d.get("binder_plddt") is not None]
n_pass = sum(1 for c in m.data["candidates"] if not c.get("is_control") and c.get("passed_filter"))
funnel = {
    "target": 1,
    "backbones": len(backbones),
    "sequences": len(designs),
    "co-folded": len(shortlist),
    "passed": n_pass,
}
fig, ax = plt.subplots(2, 2, figsize=(14, 10))
viz.plot_design_funnel(funnel, ax=ax[0, 0])
viz.plot_iptm_vs_plddt(des, ctl, ax=ax[0, 1])
viz.plot_rmsd_hist(des, ctl, ax=ax[1, 0])
viz.plot_ranked(des, by="iptm", ax=ax[1, 1])
plt.tight_layout(); plt.show()

## 12. Results, controls, and outputs

We write a reproducible **`manifest.json`** + **`candidates.csv`** for the run, and report the
success rate vs controls.

In [ ]:
m.to_csv()
summ = m.summary()
top = m.rank(by="iptm", descending=True, passed_only=False)[:10]

best_design_iptm = max((d["iptm"] for d in des), default=None)
best_ctrl_iptm = max((d["iptm"] for d in ctl), default=None)

print(f"Candidates: {summ['n_candidates']}   passed filters: {summ['n_passed']}   "
      f"controls: {summ['n_controls']}")
if summ["n_candidates"]:
    print(f"Success rate (passed / designs): {100*summ['n_passed']/summ['n_candidates']:.0f}%")
if best_design_iptm is not None and best_ctrl_iptm is not None:
    print(f"Best design ipTM/conf: {best_design_iptm:.3f}   best control: {best_ctrl_iptm:.3f}  "
          f"(designs should clearly exceed controls)")
print(f"\nTop designs by interface confidence:")
for d in top:
    s = d["scores"]
    print(f"  {d['id']:14s} iptm={s.get('iptm')}  pLDDT={s.get('binder_plddt')}  "
          f"scRMSD={s.get('self_consistency_rmsd')}  passed={d.get('passed_filter')}")
print(f"\nArtifacts written under: {run_dir}")
print(f"  manifest.json, candidates.csv, backbones/, complexes/")

## 13. Where to go next

- **Scale up.** Set `OPENHACKATHON_DEMO_MODE=0` (or raise `PBD_N_BACKBONES`, `PBD_SEQS_PER_BACKBONE`,
  `PBD_N_COFOLD`, `PBD_DIFFUSION_STEPS`) for a fuller campaign; co-folding dominates the cost.
- **Stronger interface metric.** Swap Boltz-2 for **OpenFold3** (set `BOLTZ2_URL` to an OpenFold3
  endpoint and adapt the client) to get an explicit **ipTM**.
- **Positive controls.** Re-score a *published* RBD minibinder through the same pipeline to anchor
  your success-rate thresholds.
- **Real campaigns.** Verify the epitope against the literature, add a target **MSA** for the RBD,
  filter harder, and order the top few designs for wet-lab testing.

**Responsible use.** Keep de novo binder design to legitimate research and therapeutic intent.